In [1]:
import os
import cv2
import json
import numpy as np
import pandas as pd
import networkx as nx

from tqdm import tqdm
from ultralytics import YOLO

In [2]:
model = YOLO("yolov8n.pt")

In [3]:
df = pd.read_csv(
    r"C:\Users\bhanu\Downloads\StructCompose\data\metadata\master_metadata.csv"
)

IMAGE_DIR = r"C:\Users\bhanu\Downloads\StructCompose\data\raw\CADB\images"

print(df.shape)

(9497, 13)


In [4]:
os.makedirs(
    "../data/processed/composition_features",
    exist_ok=True
)

os.makedirs(
    "../data/processed/graphs",
    exist_ok=True
)

os.makedirs(
    "../data/processed/reasoning",
    exist_ok=True
)

In [6]:
def process_image(image_path):

    img = cv2.imread(image_path)

    if img is None:
        return None

    img_rgb = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    h, w = gray.shape

    return {
        "img": img,
        "img_rgb": img_rgb,
        "gray": gray,
        "h": h,
        "w": w
    }

In [7]:
def extract_composition_features(gray):

    mean_brightness = np.mean(gray)

    std_brightness = np.std(gray)

    edges = cv2.Canny(gray, 100, 200)

    edge_density = np.sum(edges > 0) / edges.size

    return {

        "brightness_mean": float(mean_brightness),

        "brightness_std": float(std_brightness),

        "edge_density": float(edge_density)

    }

In [8]:
def detect_subjects(model, image_path):

    results = model(image_path, verbose=False)

    boxes = results[0].boxes

    detections = []

    for box in boxes:

        cls_id = int(box.cls[0])

        class_name = model.names[cls_id]

        confidence = float(box.conf[0])

        coords = box.xyxy[0].cpu().numpy()

        x1, y1, x2, y2 = coords

        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        width = x2 - x1
        height = y2 - y1

        area = width * height

        detections.append({

            "class": class_name,

            "confidence": confidence,

            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),

            "center_x": float(cx),
            "center_y": float(cy),

            "area": float(area)

        })

    return detections

In [9]:
def build_scg(detections):

    G = nx.Graph()

    for idx, det in enumerate(detections):

        G.add_node(

            idx,

            label=det["class"],

            center_x=det["center_x"],
            center_y=det["center_y"],

            area=det["area"],

            confidence=det["confidence"]

        )

    nodes = list(G.nodes())

    for i in range(len(nodes)):

        for j in range(i+1, len(nodes)):

            n1 = G.nodes[nodes[i]]
            n2 = G.nodes[nodes[j]]

            dx = n1["center_x"] - n2["center_x"]
            dy = n1["center_y"] - n2["center_y"]

            distance = np.sqrt(dx**2 + dy**2)

            G.add_edge(

                nodes[i],
                nodes[j],

                distance=float(distance)

            )

    return G

In [10]:
def composition_reasoning(G):

    if len(G.nodes()) == 0:

        return {

            "balance_score": 0,

            "subject_count": 0

        }

    x_positions = []

    for node, data in G.nodes(data=True):

        x_positions.append(data["center_x"])

    spread = np.std(x_positions)

    return {

        "subject_count": len(G.nodes()),

        "spatial_spread": float(spread)

    }

In [11]:
sample_image = df.iloc[0]["image_name"]

image_path = os.path.join(
    IMAGE_DIR,
    sample_image
)

data = process_image(image_path)

features = extract_composition_features(
    data["gray"]
)

detections = detect_subjects(
    model,
    image_path
)

G = build_scg(detections)

reasoning = composition_reasoning(G)

print(features)
print(reasoning)

{'brightness_mean': 126.44752550147156, 'brightness_std': 31.34364201801225, 'edge_density': 0.023578818783613988}
{'balance_score': 0, 'subject_count': 0}


In [12]:
subset = df.head(20)

In [13]:
subset = df

In [14]:
all_results = []

for idx, row in tqdm(subset.iterrows()):

    image_name = row["image_name"]

    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )

    try:

        data = process_image(image_path)

        if data is None:
            continue

        features = extract_composition_features(
            data["gray"]
        )

        detections = detect_subjects(
            model,
            image_path
        )

        G = build_scg(detections)

        reasoning = composition_reasoning(G)

        graph_data = nx.node_link_data(G)

        graph_save_path = (
            f"../data/processed/graphs/{image_name}.json"
        )

        with open(graph_save_path, "w") as f:

            json.dump(graph_data, f)

        result = {

            "image_name": image_name,

            **features,

            **reasoning

        }

        all_results.append(result)

    except Exception as e:

        print("Error:", image_name, e)

9497it [19:54,  7.95it/s]


In [15]:
results_df = pd.DataFrame(all_results)

results_df.head()

,image_name,brightness_mean,brightness_std,edge_density,balance_score,subject_count,spatial_spread
0,10003.jpg,126.447526,31.343642,0.023579,0.0,0,NaN
1,10007.jpg,99.889099,78.729215,0.009401,NaN,2,148.604385
2,10008.jpg,150.302874,53.160128,0.051828,0.0,0,NaN
3,1001.jpg,48.357921,57.923257,0.056185,NaN,3,119.610544
4,10010.jpg,100.815889,62.652103,0.144118,NaN,17,255.902569


In [16]:
results_df.to_csv(

    "../data/processed/full_composition_dataset.csv",

    index=False
)

print("Saved processed dataset")

Saved processed dataset
